<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">ALCHEMI CORE, MODULE 00</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Toolkit quick tour</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">See how Toolkit connects chemical systems, models, simulation controls, and execution scale before opening each part of the workflow.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Time:</b> 5 to 10 minutes<br><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> none. Familiarity with computational chemistry is preferred, and experimental researchers are welcome.<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a></div>

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Where ALCHEMI Toolkit fits</h2>
You are likely here because your research involves molecules, surfaces, materials, or bioactive compounds at the atomistic scale. Whether you simulate these systems or study them experimentally, you likely use a collection of tools that do the job, but may lose time moving between them. Many great solutions exist and serve their purpose, but the overall workflow often lacks the flexibility of a unified toolkit.

**ALCHEMI Toolkit** was inspired by a common challenge in computational chemistry: structure preparation and data loading often happen on the CPU, while models and simulations run on the GPU. As work passes through each stage, the same atomic data must be transferred, converted, and synchronized.

ALCHEMI Toolkit gives these systems a shared, **GPU-resident** data interface and **accelerated building blocks** that bring **AI models**, physical simulations, and scalable execution into one workflow that supports your R&D needs and scientific curiosity.

The sketch below shows the full cycle, with illustrative numbers:


With Toolkit the whole simulation cycle stays on the GPU: the batch is built, run and handled there.

<img src="assets/core00-gpu-cycle-light.svg" alt="Two loops compared. Before: the batch is built on the CPU, copied to the GPU for the model and copied back. With ALCHEMI: the batch is built, run and handled on the GPU." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

In [ ]:
# Load PyTorch and the focused helper interface used in this quick tour.
import torch
from helpers import module00 as helpers

In [ ]:
# Apply the shared notebook style, then choose GPU if available
helpers.configure_tutorial()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
if device.type == "cuda":
    print("gpu:   ", torch.cuda.get_device_name(device))

Pick up to five systems you care about. You will relax them together in the next section.

In [ ]:
#  Choose up to five systems of interest
research_systems = helpers.opening_performance_catalog()
research_picker = helpers.show_structure_explorer(research_systems)
research_picker

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">2. Relax the selected structures together</h2>

Toolkit packs your selection into one `Batch` and relaxes every system with [MACE-MPA-0](https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html), a machine-learned interatomic potential (MLIP) that predicts energies and forces. Each structure starts with its atoms displaced at random by about 0.2 Å, so there is something to relax.

The next cell repeats your selection `batch_multiplier` times (default 12) inside one `Batch` and relaxes it with L-BFGS; each system stops as soon as it converges, so Toolkit relaxes a larger mixed workload at once. The player keeps the result easy to follow with one trajectory for each system you chose.

In [ ]:
batch_multiplier = 12

relaxation = helpers.relax_selected_structures(
    research_picker,
    device=device,
    fmax=0.05,  # eV/Å, each system stops here
    max_steps=500 if device.type == "cuda" else 4,
    distortion_std_angstrom=0.20,
    distortion_seed=7,
    batch_multiplier=batch_multiplier,
)
relaxation["player"]

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">3. Put the run time in perspective</h2>
The same work can be done one structure at a time, as most codes do. The left graph sets your batch beside that over 50 steps, its time taken from the rate of your run. The right graph shows what larger batches reach on larger GPUs, for five reference molecules with cuEquivariance and compilation on; your batch, the dot, ran with neither.

In [ ]:
helpers.plot_relaxation_time_perspective(relaxation)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">4. Designed for two modes</h2>
<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 16px 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 300px; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; background: transparent; color: inherit;"><div style="display: flex; align-items: center; gap: 8px; font-size: 20px; font-weight: 500; line-height: 1.25;"><img src="assets/fork-self-light.svg" alt="" width="22" height="22" style="display: block; flex: none; width: 22px; height: 22px; margin: 0;"><span>User driving manually</span></div><div style="margin: 6px 0 0; font-size: 15px; opacity: 0.85;"><span style="color: #76b900;">●</span>&nbsp; Best for: finer manual control</div><div style="margin: 6px 0 0; font-size: 15px; opacity: 0.85;"><span style="color: #76b900;">●</span>&nbsp; Experts and developers</div><div style="margin: 12px 0 0; font-size: 14px;"><a href="https://nvidia.github.io/nvalchemi-toolkit/" style="font-weight: 600; color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Docs</a><span style="display: inline-block; width: 18px;"></span><a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="font-weight: 600; color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Examples</a></div></div><div style="flex: 1 1 300px; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; background: transparent; color: inherit;"><div style="display: flex; align-items: center; gap: 8px; font-size: 20px; font-weight: 500; line-height: 1.25;"><img src="assets/fork-agent-light.svg" alt="" width="22" height="22" style="display: block; flex: none; width: 22px; height: 22px; margin: 0;"><span>Researcher directing an agent</span></div><div style="margin: 6px 0 0; font-size: 15px; opacity: 0.85;"><span style="color: #76b900;">●</span>&nbsp; Best for: fast prototyping</div><div style="margin: 6px 0 0; font-size: 15px; opacity: 0.85;"><span style="color: #76b900;">●</span>&nbsp; Computational and experimental researchers</div><div style="margin: 12px 0 0; font-size: 14px;"><a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="font-weight: 600; color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Agent skills</a><span style="display: inline-block; width: 18px;"></span><a href="https://developer.nvidia.com/blog/how-ai-coding-agents-can-unlock-materials-simulation-with-nvidia-alchemi-toolkit/" style="font-weight: 600; color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Blog: Toolkit workflows with AI coding agents</a></div></div></div>

Toolkit is built for two ways of working. An expert can drive it directly, with the documentation and the examples at hand. A researcher can direct a coding agent, which writes the Toolkit calls from the published skills.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">5. What you can build with Toolkit</h2>
The relaxation connected the complete path, each structure held as AtomicData before it joins the Batch:

`structures → AtomicData → Batch → model (MACE) → relaxation (L-BFGS) → final results`

The relaxation above followed one possible Toolkit workflow, but ALCHEMI Toolkit is so much more. It lets you connect your own data, choose or build models, create simulation methods, and add the controls and reporting your study needs.

**What parts of an atomistic workflow can you use, combine, or create yourself?**

<img src="assets/core00-step-by-step-light.svg" alt="Five molecules followed through the Toolkit: one batch, a model with added physics, relaxation and dynamics, then reaction paths, enhanced sampling and training; hooks and inflight refill act at every step, an agent can drive every step, and Toolkit-Ops kernels run underneath." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0 0 16px;">

**Bottom line:** Toolkit gives you ready-to-use components and blocks to add your own tools via a unified GPU-accelerated interface.

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

Toolkit is part of NVIDIA ALCHEMI. Explore the connected libraries, services, and technical guides below.

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76B900; border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5;"><img src="assets/explore-agents-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>★ <a href="https://developer.nvidia.com/blog/how-ai-coding-agents-can-unlock-materials-simulation-with-nvidia-alchemi-toolkit/">Build Toolkit workflows with AI coding agents</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Blog about what coding agents get right, and what they miss.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Toolkit workflows:</b></div><div style="margin: 2px 0 0;"><span style="white-space: nowrap;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit">Repository</a></span><span style="display: inline-block; width: 14px;"></span><span style="white-space: nowrap;"><a href="https://nvidia.github.io/nvalchemi-toolkit/">Docs</a></span></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Model wrappers, data management, model training, simulation tools - GPU native and scalable.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5;"><img src="assets/explore-ops-wrench-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Toolkit-Ops:</b></div><div style="margin: 2px 0 0;"><span style="white-space: nowrap;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit-ops">Repository</a></span><span style="display: inline-block; width: 14px;"></span><span style="white-space: nowrap;"><a href="https://nvidia.github.io/nvalchemi-toolkit-ops/">Docs</a></span></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Accelerated batched neighbor lists, dispersion, and electrostatics.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5;"><img src="assets/explore-nim-standard-light.png" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>ALCHEMI NIM:</b></div><div style="margin: 2px 0 0;"><span style="white-space: nowrap;"><a href="https://docs.nvidia.com/nim/alchemi/alchemi-bgr/latest/index.html">Documentation</a></span><span style="display: inline-block; width: 14px;"></span><span style="white-space: nowrap;"><a href="https://developer.nvidia.com/blog/faster-chemistry-and-materials-discovery-with-ai-powered-simulations-using-nvidia-alchemi/">Technical blog</a></span></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Packaged batched relaxation and molecular dynamics services.</div></div></div>


<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Module 00 recap</h2><div style="margin: 0 0 10px;">You chose systems, relaxed them together in one <code>Batch</code>, and saw how Toolkit connects data, models, simulation controls, and scale.</div><div style="margin: 0;"><b>Choose your path:</b></div><ul style="margin: 4px 0 0; padding-left: 0; list-style: none;"><li><img src="assets/fork-self-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;">Drive Toolkit yourself: <a href="alchemi-core-01-data-and-batching.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Module 01: Toolkit fundamentals and data structures →</a></li><li><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;">Work with a coding agent: <a href="alchemi-agent-01-meet-your-agent.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Meet your agent →</a></li></ul></div>